# CIFAR-10 SHViT Baseline — From Scratch

Single-head Vision Transformer baseline with standard LayerNorm and Softmax.

- CIFAR-10: 45,000 training, 5,000 validation, and 10,000 test images
- SHViT: 4×4 patches, embedding dimension 128, depth 3, one attention head
- Seeds: 42, 886, 1024, 1991, and 2333
- Each seed writes to its own lowercase directory under `runs/`


In [1]:
import gc
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


EXPERIMENT_NAME = "1_cifar10_shvit_baseline"
FROM_PRETRAINED = False
USE_LENS_NORM = False
USE_LENS_MAX = False

SEEDS = [42, 886, 1024, 1991, 2333]
DATA_ROOT = Path("./data")
RUNS_ROOT = Path("./runs")

IMAGE_SIZE = 32
PATCH_SIZE = 4
NUM_CLASSES = 10
EMBED_DIM = 128
DEPTH = 3
MLP_RATIO = 4.0
DROPOUT = 0.1
ATTENTION_DROPOUT = 0.0

LENS_NORM_HIDDEN = 32
LENS_MAX_ROW_HIDDEN = 32
LENS_MAX_ELEMENT_HIDDEN = 16

BATCH_SIZE = 256
NUM_WORKERS = 4
VALIDATION_SIZE = 5000
EPOCHS = 400
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.05
GRAD_CLIP_NORM = 1.0

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"device={DEVICE} | experiment={EXPERIMENT_NAME}")


/home/yuhao/anaconda3/envs/brevitas/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cuda:0 | experiment=1_cifar10_shvit_baseline


In [2]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class LENSNorm(nn.Module):
    """Predicts one affine scale and bias for every token."""

    def __init__(self, dim, hidden_dim=32, scale_min=0.05, scale_max=12.0):
        super().__init__()
        self.scale_min = scale_min
        self.scale_max = scale_max
        self.predictor_0 = nn.Linear(dim, hidden_dim)
        self.predictor_1 = nn.Linear(hidden_dim, 2)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_normal_(self.predictor_0.weight, nonlinearity="relu")
        nn.init.zeros_(self.predictor_0.bias)
        nn.init.normal_(self.predictor_1.weight, mean=0.0, std=1e-3)
        with torch.no_grad():
            self.predictor_1.bias.copy_(torch.tensor([1.0, 0.0]))

    def forward(self, x):
        affine = self.predictor_1(F.relu(self.predictor_0(x)))
        scale = affine[..., 0:1].clamp(self.scale_min, self.scale_max)
        bias = affine[..., 1:2]
        return scale * x + bias


class LENSMax(nn.Module):
    """Maps each complete attention-score row to positive attention weights."""

    def __init__(self, num_tokens, row_hidden=32, element_hidden=16):
        super().__init__()
        self.num_tokens = num_tokens
        self.row_mlp = nn.Sequential(
            nn.Linear(num_tokens, row_hidden),
            nn.ReLU(),
            nn.Linear(row_hidden, 1),
        )
        self.element_mlp = nn.Sequential(
            nn.Linear(2, element_hidden),
            nn.ReLU(),
            nn.Linear(element_hidden, 1),
        )
        self.reset_parameters()

    def reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                nn.init.zeros_(module.bias)

    def forward(self, scores):
        batch_size, num_queries, num_keys = scores.shape
        if num_keys != self.num_tokens:
            raise ValueError(f"LENSMax expects {self.num_tokens} keys, received {num_keys}")

        flat_rows = scores.reshape(batch_size * num_queries, num_keys)
        row_state = self.row_mlp(flat_rows)
        row_state = row_state.unsqueeze(1).expand(-1, num_keys, -1)
        element_input = torch.cat((flat_rows.unsqueeze(-1), row_state), dim=-1)
        logits = self.element_mlp(element_input).squeeze(-1)

        divisor = float(2 ** int(math.floor(math.log2(num_keys))))
        weights = torch.sigmoid(logits) / divisor
        return weights.reshape(batch_size, num_queries, num_keys)


In [3]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_tokens, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.scale = dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.lens_max = None

    def forward(self, x):
        batch_size, num_tokens, dim = x.shape
        qkv = self.qkv(x).reshape(batch_size, num_tokens, 3, dim).permute(2, 0, 1, 3)
        q, k, v = qkv.unbind(0)
        scores = (q @ k.transpose(-2, -1)) * self.scale

        if self.lens_max is None:
            attention = F.softmax(scores, dim=-1)
        else:
            attention = self.lens_max(scores)

        attention = self.attn_drop(attention)
        x = attention @ v
        x = self.proj(x)
        return self.proj_drop(x)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, drop=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        return self.drop(self.fc2(x))


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_tokens, mlp_ratio=4.0, drop=0.0, attn_drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = SingleHeadSelfAttention(
            dim=dim,
            num_tokens=num_tokens,
            attn_drop=attn_drop,
            proj_drop=drop,
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = FeedForward(dim, int(dim * mlp_ratio), drop=drop)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class CIFAR10SHViT(nn.Module):
    def __init__(
        self,
        image_size=32,
        patch_size=4,
        num_classes=10,
        dim=128,
        depth=3,
        mlp_ratio=4.0,
        drop=0.1,
        attn_drop=0.0,
    ):
        super().__init__()
        self.dim = dim
        self.num_patches = (image_size // patch_size) ** 2
        self.num_tokens = self.num_patches + 1
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_tokens, dim))
        self.pos_drop = nn.Dropout(drop)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, self.num_tokens, mlp_ratio, drop, attn_drop)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, mode="fan_out")
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls_token = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])


def install_lens_norm(model):
    for block in model.blocks:
        block.norm1 = LENSNorm(model.dim, hidden_dim=LENS_NORM_HIDDEN)
        block.norm2 = LENSNorm(model.dim, hidden_dim=LENS_NORM_HIDDEN)
    model.norm = LENSNorm(model.dim, hidden_dim=LENS_NORM_HIDDEN)


def install_lens_max(model):
    for block in model.blocks:
        block.attn.lens_max = LENSMax(
            num_tokens=model.num_tokens,
            row_hidden=LENS_MAX_ROW_HIDDEN,
            element_hidden=LENS_MAX_ELEMENT_HIDDEN,
        )


def checkpoint_model_state(checkpoint):
    return checkpoint["model"] if "model" in checkpoint else checkpoint


def build_experiment_model(seed):
    model = CIFAR10SHViT(
        image_size=IMAGE_SIZE,
        patch_size=PATCH_SIZE,
        num_classes=NUM_CLASSES,
        dim=EMBED_DIM,
        depth=DEPTH,
        mlp_ratio=MLP_RATIO,
        drop=DROPOUT,
        attn_drop=ATTENTION_DROPOUT,
    )

    if FROM_PRETRAINED:
        baseline_path = RUNS_ROOT / f"1_cifar10_shvit_baseline_seed{seed}" / "best.pt"
        if not baseline_path.exists():
            raise FileNotFoundError(f"Missing baseline checkpoint: {baseline_path}")
        checkpoint = torch.load(baseline_path, map_location="cpu")
        model.load_state_dict(checkpoint_model_state(checkpoint), strict=True)

    if USE_LENS_NORM:
        install_lens_norm(model)
    if USE_LENS_MAX:
        install_lens_max(model)

    return model


In [4]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)


train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])


def make_loaders(seed):
    train_dataset = datasets.CIFAR10(
        root=DATA_ROOT,
        train=True,
        download=True,
        transform=train_transform,
    )
    validation_dataset = datasets.CIFAR10(
        root=DATA_ROOT,
        train=True,
        download=False,
        transform=eval_transform,
    )
    test_dataset = datasets.CIFAR10(
        root=DATA_ROOT,
        train=False,
        download=True,
        transform=eval_transform,
    )

    split_generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(train_dataset), generator=split_generator).tolist()
    validation_indices = indices[:VALIDATION_SIZE]
    training_indices = indices[VALIDATION_SIZE:]

    training_subset = Subset(train_dataset, training_indices)
    validation_subset = Subset(validation_dataset, validation_indices)
    loader_generator = torch.Generator().manual_seed(seed)
    pin_memory = DEVICE.type == "cuda"

    train_loader = DataLoader(
        training_subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        persistent_workers=NUM_WORKERS > 0,
        generator=loader_generator,
    )
    validation_loader = DataLoader(
        validation_subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        persistent_workers=NUM_WORKERS > 0,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        persistent_workers=NUM_WORKERS > 0,
    )
    return train_loader, validation_loader, test_loader


In [5]:
@torch.no_grad()
def evaluate(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, targets in data_loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = F.cross_entropy(logits, targets)
        total_loss += loss.item() * targets.size(0)
        total_correct += (logits.argmax(dim=1) == targets).sum().item()
        total_examples += targets.size(0)

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


def optimizer_to_device(optimizer):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(DEVICE)


def save_json(path, value):
    path.write_text(json.dumps(value, indent=2), encoding="utf-8")


def train_model(model, train_loader, validation_loader, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    best_path = output_dir / "best.pt"
    last_path = output_dir / "last.pt"
    history_path = output_dir / "history.json"
    log_path = output_dir / "train.log"

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    start_epoch = 1
    best_validation_accuracy = -1.0
    history = []

    if last_path.exists():
        checkpoint = torch.load(last_path, map_location="cpu")
        model.load_state_dict(checkpoint_model_state(checkpoint), strict=True)
        optimizer.load_state_dict(checkpoint["optimizer"])
        optimizer_to_device(optimizer)
        scheduler.load_state_dict(checkpoint["scheduler"])
        start_epoch = checkpoint["epoch"] + 1
        best_validation_accuracy = checkpoint["best_validation_accuracy"]
        history = checkpoint.get("history", [])

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        epoch_start = time.time()
        total_loss = 0.0
        total_correct = 0
        total_examples = 0

        for images, targets in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = F.cross_entropy(logits, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

            total_loss += loss.item() * targets.size(0)
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_examples += targets.size(0)

        validation_metrics = evaluate(model, validation_loader)
        learning_rate = optimizer.param_groups[0]["lr"]
        scheduler.step()

        epoch_metrics = {
            "epoch": epoch,
            "learning_rate": learning_rate,
            "train_loss": total_loss / total_examples,
            "train_accuracy": total_correct / total_examples,
            "validation_loss": validation_metrics["loss"],
            "validation_accuracy": validation_metrics["accuracy"],
            "seconds": time.time() - epoch_start,
        }
        history.append(epoch_metrics)

        checkpoint = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "epoch": epoch,
            "best_validation_accuracy": max(
                best_validation_accuracy,
                validation_metrics["accuracy"],
            ),
            "history": history,
        }
        torch.save(checkpoint, last_path)

        if validation_metrics["accuracy"] > best_validation_accuracy:
            best_validation_accuracy = validation_metrics["accuracy"]
            checkpoint["best_validation_accuracy"] = best_validation_accuracy
            torch.save(checkpoint, best_path)

        save_json(history_path, history)
        message = (
            f"Epoch {epoch:03d}/{EPOCHS} | lr={learning_rate:.3e} | "
            f"train loss={epoch_metrics['train_loss']:.4f} "
            f"acc={epoch_metrics['train_accuracy']:.4f} | "
            f"val loss={validation_metrics['loss']:.4f} "
            f"acc={validation_metrics['accuracy']:.4f} | "
            f"time={epoch_metrics['seconds'] / 60:.1f}m"
        )
        print(message)
        with log_path.open("a", encoding="utf-8") as log_file:
            log_file.write(message + "\n")

    return best_path


In [6]:
results = []

for seed in SEEDS:
    print(f"\n=== {EXPERIMENT_NAME} | seed={seed} ===")
    seed_everything(seed)
    output_dir = RUNS_ROOT / f"{EXPERIMENT_NAME}_seed{seed}"
    train_loader, validation_loader, test_loader = make_loaders(seed)
    model = build_experiment_model(seed).to(DEVICE)

    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    trainable_parameters = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    print(f"parameters={total_parameters:,} | trainable={trainable_parameters:,}")

    best_path = train_model(model, train_loader, validation_loader, output_dir)
    best_checkpoint = torch.load(best_path, map_location="cpu")
    model.load_state_dict(checkpoint_model_state(best_checkpoint), strict=True)
    model.to(DEVICE)

    validation_metrics = evaluate(model, validation_loader)
    test_metrics = evaluate(model, test_loader)
    seed_result = {
        "seed": seed,
        "run_directory": str(output_dir),
        "best_epoch": best_checkpoint["epoch"],
        "validation": validation_metrics,
        "test": test_metrics,
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
    }
    save_json(output_dir / "metrics.json", seed_result)
    results.append(seed_result)
    print(
        f"best epoch={seed_result['best_epoch']} | "
        f"val acc={validation_metrics['accuracy']:.4f} | "
        f"test acc={test_metrics['accuracy']:.4f}"
    )

    del model, train_loader, validation_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


test_accuracies = np.array([result["test"]["accuracy"] for result in results])
summary = {
    "experiment": EXPERIMENT_NAME,
    "seeds": SEEDS,
    "results": results,
    "test_accuracy_mean": float(test_accuracies.mean()),
    "test_accuracy_std": float(test_accuracies.std(ddof=1)),
}
save_json(RUNS_ROOT / f"{EXPERIMENT_NAME}_summary.json", summary)
print(
    f"\n{EXPERIMENT_NAME}: "
    f"test accuracy={summary['test_accuracy_mean']:.4f} "
    f"± {summary['test_accuracy_std']:.4f} over {len(SEEDS)} seeds"
)



=== 1_cifar10_shvit_baseline | seed=42 ===


Files already downloaded and verified


Files already downloaded and verified


parameters=611,082 | trainable=611,082


Epoch 032/400 | lr=9.853e-04 | train loss=0.8240 acc=0.7077 | val loss=0.8332 acc=0.7152 | time=0.1m


Epoch 033/400 | lr=9.843e-04 | train loss=0.8176 acc=0.7107 | val loss=0.7904 acc=0.7228 | time=0.1m


... (1,985 outputs left out)

Epoch 399/400 | lr=6.168e-08 | train loss=0.2120 acc=0.9221 | val loss=0.7777 acc=0.8030 | time=0.0m


Epoch 400/400 | lr=1.542e-08 | train loss=0.2163 acc=0.9216 | val loss=0.7777 acc=0.8030 | time=0.0m


best epoch=250 | val acc=0.8112 | test acc=0.8014



1_cifar10_shvit_baseline: test accuracy=0.8033 ± 0.0028 over 5 seeds
